<a href="https://colab.research.google.com/github/AmirJlr/Thesis/blob/master/examples/Lipo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install torch_geometric
!pip install deepchem
!pip install rdkit
!pip install torchinfo
!pip install molfeat

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 83.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 84.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 67.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 79.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
import os
os.chdir('Thesis')

In [4]:
!pwd

/content/Thesis


In [5]:
!ls

best_models  documents	figs	modules    README.md	     test_results
data	     examples	models	notebooks  requirements.txt  Visualization


In [7]:
import random
import numpy as np
import torch

SEED = 121
def seed_set(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_set(SEED)

In [8]:
%load modules/data_handler.py

In [9]:
from modules.data_handler import FingerprintsDescriptorsCalculator, PCAReducer, DTsetBasic

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [10]:
# Usage Example :
import pandas as pd

df = pd.read_csv('/content/Lipophilicity.csv')
smiles_column = df['smiles'].values

calculator = FingerprintsDescriptorsCalculator(smiles_column)

ecfp = calculator.calculate_ecfp()
topological = calculator.calculate_topological()
maccs = calculator.calculate_maccs()
estate = calculator.calculate_estate()
rdkit2D = calculator.calculate_rdkit2D()
phar2D = calculator.calculate_phar2D()
invalid_indices = calculator.get_invalid_indices()

0it [00:00, ?it/s]

/usr/local/lib/python3.10/dist-packages/molfeat/calc/descriptors.py:46: RuntimeWarning: All-NaN slice encountered
  min_charge, max_charge = np.nanmin(atomic_charges), np.nanmax(atomic_charges)


In [11]:
invalid_indices

[]

In [12]:
# Usage Example :
N_COMPONENTS = 64
reducer = PCAReducer(n_components=N_COMPONENTS)

ecfp_reduced = reducer.reduce_ecfp(ecfp)
topological_reduced = reducer.reduce_topological(topological)
maccs_reduced = reducer.reduce_maccs(maccs)
estate_reduced = reducer.reduce_estate(estate)
rdkit2D_reduced = reducer.reduce_rdkit2D(rdkit2D)
phar2D_reduced = reducer.reduce_phar2D(phar2D)

# phar3D_reduced = reducer.reduce_phar3D(phar3D)
# rdkit3D_reduced = reducer.reduce_rdkit3D(rdkit3D)

In [13]:
directory = 'data/basic-64/raw'
CSV_PATH = 'data/basic-64/raw/lipo_cleaned.csv'

if not os.path.exists(directory):
    os.makedirs(directory)

df.drop(invalid_indices).to_csv(CSV_PATH)

In [14]:
dataset_64 = DTsetBasic(root='data/basic-64', filename='lipo_cleaned.csv', smiles_column='smiles', label_column='exp',
    ECFP=ecfp_reduced, Topological=topological_reduced, MACCS=maccs_reduced,
    EState=estate_reduced, Rdkit2D=rdkit2D_reduced, Phar2D=phar2D_reduced)

Processing...


0it [00:00, ?it/s]

Done!


In [15]:
dataset_64[0]

Data(x=[24, 9], edge_index=[2, 54], edge_attr=[54, 3], smiles='Cn1c(CN2CCN(CC2)c3ccc(Cl)cc3)nc4ccccc14', y=[1, 1], ECFP=[1, 64], Topological=[1, 64], MACCS=[1, 64], EState=[1, 64], Rdkit2D=[1, 64], Phar2D=[1, 64])

In [16]:
from modules.data_handler import load_and_process_data

In [17]:
train_loader_DTsetBasic, valid_loader_DTsetBasic, test_loader_DTsetBasic = load_and_process_data(dataset_64, test_size=0.2)

In [18]:
%load modules/utils_regression.py
from modules.utils_regression import run_epoch_reg, train_reg

## HybridModel

In [25]:
%load models/GnnPooling.py
from models.GnnPooling import GINGAT

In [26]:
import torch
from torchinfo import summary

EPOCHS = 100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### GIN-Attention-GAT

In [27]:
LOSS_FUNCTION = torch.nn.MSELoss()
model_GinAttGat = GINGAT(node_dim=9, edge_dim=3, hidden_channels=96, out_channels=N_COMPONENTS, heads=8, dropout=0.2, pooling_type='attention', num_tasks=1)
optimizer_GinAttGat = torch.optim.Adam(model_GinAttGat.parameters(), lr=0.002)

summary(model_GinAttGat)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  960
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  9,312
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  192
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  9,312
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  9,312
│    └─Linear: 2-7                       384
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  192
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [28]:
results_GinAttGat = train_reg(model = model_GinAttGat,
    optimizer = optimizer_GinAttGat,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader_DTsetBasic,
    val_loader = valid_loader_DTsetBasic,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "GIN-Attention-GAT")

Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch: 001, Train loss: 1.9310, Train RMSE: 1.3910, Val loss: 1.3717, Val RMSE: 1.1728


Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch: 099, Train loss: 0.5316, Train RMSE: 0.7283, Val loss: 0.7889, Val RMSE: 0.8880


Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch: 100, Train loss: 0.5017, Train RMSE: 0.7085, Val loss: 0.5697, Val RMSE: 0.7535


### GIN-LSTM-GAT

In [29]:
LOSS_FUNCTION = torch.nn.MSELoss()
model_GinLstmGat = GINGAT(node_dim=9, edge_dim=3, hidden_channels=96, out_channels=N_COMPONENTS, heads=6, dropout=0.2, pooling_type='lstm', num_tasks=1)
optimizer_GinLstmGat = torch.optim.Adam(model_GinLstmGat.parameters(), lr=0.002)

summary(model_GinLstmGat)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  960
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  9,312
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  192
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  9,312
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  9,312
│    └─Linear: 2-7                       384
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  192
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [30]:
results_GinLstmGat = train_reg(model = model_GinLstmGat,
    optimizer = optimizer_GinLstmGat,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader_DTsetBasic,
    val_loader = valid_loader_DTsetBasic,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "GIN-LSTM-GAT")

Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch: 001, Train loss: 2.1443, Train RMSE: 1.4656, Val loss: 1.3959, Val RMSE: 1.1834


Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

Iteration:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch: 097, Train loss: 0.2364, Train RMSE: 0.4868, Val loss: 0.7876, Val RMSE: 0.8896


Iteration:   0%|          | 0/95 [00:00<?, ?it/s]

KeyboardInterrupt: 

### GIN-GRU-GAT

In [ ]:
LOSS_FUNCTION = torch.nn.MSELoss()
model_GinGruGat = GINGAT(node_dim=9, edge_dim=3, hidden_channels=96, out_channels=N_COMPONENTS, heads=6, dropout=0.2, pooling_type='gru', num_tasks=1)
optimizer_GinGruGat = torch.optim.Adam(model_GinGruGat.parameters(), lr=0.002)

summary(model_GinGruGat)

In [ ]:
results_GinGruGat = train_reg(model = model_GinGruGat,
    optimizer = optimizer_GinGruGat,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader_DTsetBasic,
    val_loader = valid_loader_DTsetBasic,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "GIN-GRU-GAT")

## Test Results

In [ ]:
import numpy as np
import torch
from torch import device
from torch.utils.data import DataLoader
from torch.nn import Linear
import torch.nn.functional as F
from torch.nn import MSELoss
from torch.utils.tensorboard import SummaryWriter
from torch.optim import Adam

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from copy import deepcopy
from math import sqrt
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import os

### Best Global Attention Model On Test Data :

In [ ]:
results_GinAttGat

In [ ]:
best_model_att = results_GinAttGat['best_model']

_ , test_rmse_att = run_epoch_reg(model = best_model_att, optimizer=None, data_loader=test_loader_DTsetBasic,
                                    loss_function=torch.nn.MSELoss(), device='cpu',
                                    edge_attr=True, pass_data=True)

print(f'Test RMSE: {test_rmse_att:.4f}')

In [ ]:
# Save the best model
# torch.save(best_model_att.state_dict(), 'lipo_model_att.pth')

### Best GRU-Based Model On Test Data :

In [ ]:
results_GinGruGat

In [ ]:
best_model_gru = results_GinGruGat['best_model']
_ , test_rmse_gru = run_epoch_reg(model = best_model_gru, optimizer=None, data_loader=test_loader_DTsetBasic,
                                    loss_function=torch.nn.MSELoss(), device='cpu',
                                    edge_attr=True, pass_data=True)

print(f'Test RMSE: {test_rmse_gru:.4f}')

In [ ]:
# Save the best model
# torch.save(best_model_gru.state_dict(), 'lipo_model_gru.pth')

### Best LSTM-Based Model On Test Data :

In [ ]:
results_GinLstmGat

In [ ]:
best_model_lstm = results_GinLstmGat['best_model']
_ , test_rmse_lstm = run_epoch_reg(model = best_model_lstm, optimizer=None, data_loader=test_loader_DTsetBasic,
    loss_function=torch.nn.MSELoss(), device='cpu',
    edge_attr=True, pass_data=True)

print(f'Test RMSE: {test_rmse_lstm:.4f}')

In [ ]:
# Save the best model
# torch.save(best_model_lstm.state_dict(), 'lipo_model_lstm.pth')

## Compare Models :

In [ ]:
### Use TensorBoard for compare metrics ###

from torch.utils.tensorboard import SummaryWriter

%load_ext tensorboard

%tensorboard --logdir runs